In [1]:
from sklearn import svm
import pandas as pd
from sklearn.model_selection import cross_validate, train_test_split,GridSearchCV
from sklearn.metrics import make_scorer, accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [2]:
def fusionner_dataframes(liste_df):
    # Fusionner les DataFrames de la liste en utilisant la colonne 'a'
    merged_df = liste_df[0]  # Initialiser avec le premier DataFrame de la list
    for df in liste_df[1:]:
        merged_df = pd.merge(merged_df, df, on=['gameId', 'HOME_WON', 'ELO', 'ELO_PROB'])
    return merged_df

In [3]:
data = pd.read_csv('../dataset/final_dataset_diff_f_L10.csv', parse_dates=['GAME_DATE'], dtype={'gameId' : str, 'H_teamId' : str, 'A_teamId' : str,})
data = data.round(2)

In [4]:
condition = (data['GAME_DATE'] > pd.to_datetime('2023-09-01')) & (data['GAME_DATE'] < pd.to_datetime('2024-09-01'))
data_test = data[condition]
#data_test = data_test.drop(columns=['GAME_DATE','gameId', 'A_teamId', 'H_teamId'])
data_train = data[~condition]
#data_train = data_train.drop(columns=['GAME_DATE','gameId', 'A_teamId', 'H_teamId'])

In [5]:
X_train = data_train.drop(columns=['HOME_WON', 'GAME_DATE','gameId', 'A_teamId', 'H_teamId','H_teamId','A_teamId','H_POINTS', 'A_POINTS','H_teamName', 'A_teamName', 'trueShootingPercentage_L10'])  # Fonctionnalités
y_train = data_train['HOME_WON']  # Cible

X_test = data_test.drop(columns=['HOME_WON', 'GAME_DATE','gameId', 'A_teamId', 'H_teamId','H_teamId','A_teamId','H_POINTS', 'A_POINTS','H_teamName', 'A_teamName', 'trueShootingPercentage_L10'])  # Fonctionnalités
y_test = data_test['HOME_WON']

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)
X_test

In [6]:
data

In [7]:
#svm_classifier = svm.SVC(kernel='rbf', C=0.05, gamma=0.45)
# Initialiser le modèle k-NN
svm_classifier = svm.SVC()

# Définir la grille de paramètres à tester
param_grid = {
    'kernel': ['linear'],
    'C': [0.150, 0.2],
    'gamma': [0.1, 0.5]
}

In [8]:
# Utiliser GridSearchCV pour trouver les meilleures combinaisons de paramètres
grid_search = GridSearchCV(svm_classifier, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

In [9]:
# Afficher les meilleurs paramètres trouvés
print("Meilleurs paramètres trouvés : ", grid_search.best_params_)
print("Meilleure accuracy moyenne sur l'ensemble d'entraînement : ", grid_search.best_score_)

In [10]:
y_pred = grid_search.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)  # y_test sont les étiquettes de classe réelles des données de test
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("F1-score:", f1)
print("Matrice de confusion :")
print(conf_matrix)

In [11]:
# Afficher les scores de performance pour chaque fold
#print("Scores de validation croisée - Accuracy : ", cv_results['test_accuracy'])
#print("Scores de validation croisée - F1 : ", cv_results['test_f1'])

# Calculer la moyenne des scores de performance
#mean_accuracy = cv_results['test_accuracy'].mean()
#mean_f1 = cv_results['test_f1'].mean()
#print("Accuracy moyenne avec validation croisée : {:.2f}%".format(mean_accuracy * 100))
#print("F1 moyenne avec validation croisée : {:.2f}".format(mean_f1))